# LightGCN Baseline (Implicit CF)
Run cells top-to-bottom. Uses the same fixed train/test split from phase5_outputs.

In [1]:
# Install dependencies (Colab)
!pip install -q numpy scipy scikit-learn torch

import csv
import json
import math
import os
from datetime import datetime
import numpy as np
import torch
from scipy import sparse

def load_sparse_matrix(path: str) -> sparse.csr_matrix:
    matrix = sparse.load_npz(path)
    if not sparse.isspmatrix_csr(matrix):
        matrix = matrix.tocsr()
    return matrix

def load_json(path: str) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def _top_k_indices(scores: np.ndarray, k: int) -> np.ndarray:
    if k <= 0:
        return np.array([], dtype=np.int64)
    if k >= scores.shape[0]:
        return np.argsort(-scores)
    idx = np.argpartition(-scores, k - 1)[:k]
    return idx[np.argsort(-scores[idx])]

def compute_user_metrics(
    scores: np.ndarray,
    train_items: np.ndarray,
    test_items: np.ndarray,
    k: int,
    use_ndcg: bool = True,
):
    if test_items.size == 0 or k <= 0 or scores.size == 0:
        return None
    scores = np.asarray(scores, dtype=float)
    scores = scores.copy()
    scores[train_items] = -np.inf
    k = min(k, scores.size)
    ranked = _top_k_indices(scores, k)
    test_set = set(test_items.tolist())
    hits = sum(1 for item in ranked if item in test_set)
    precision = hits / k
    recall = hits / len(test_set)
    ndcg = 0.0
    if use_ndcg:
        dcg = 0.0
        for rank, item in enumerate(ranked):
            if item in test_set:
                dcg += 1.0 / math.log2(rank + 2)
        ideal = min(len(test_set), k)
        idcg = sum(1.0 / math.log2(rank + 2) for rank in range(ideal))
        ndcg = (dcg / idcg) if idcg > 0 else 0.0
    return precision, recall, ndcg

def evaluate_ranking(
    score_fn,
    train_interactions: sparse.csr_matrix,
    test_interactions: sparse.csr_matrix,
    k: int,
    use_ndcg: bool = True,
) -> dict:
    precisions = []
    recalls = []
    ndcgs = []
    num_users = train_interactions.shape[0]
    for user_id in range(num_users):
        test_items = test_interactions[user_id].indices
        if test_items.size == 0:
            continue
        train_items = train_interactions[user_id].indices
        scores = score_fn(user_id)
        metrics = compute_user_metrics(scores, train_items, test_items, k, use_ndcg)
        if metrics is None:
            continue
        precision, recall, ndcg = metrics
        precisions.append(precision)
        recalls.append(recall)
        if use_ndcg:
            ndcgs.append(ndcg)
    result = {
        "k": float(k),
        "users_evaluated": float(len(precisions)),
        "precision_at_k": float(np.mean(precisions)) if precisions else 0.0,
        "recall_at_k": float(np.mean(recalls)) if recalls else 0.0,
    }
    if use_ndcg:
        result["ndcg_at_k"] = float(np.mean(ndcgs)) if ndcgs else 0.0
    return result

def recommend_top_k(scores: np.ndarray, train_items: np.ndarray, k: int) -> np.ndarray:
    scores = np.asarray(scores, dtype=float)
    scores = scores.copy()
    scores[train_items] = -np.inf
    k = min(k, scores.size)
    return _top_k_indices(scores, k)

def build_norm_adj(train_coo: sparse.coo_matrix, num_users: int, num_items: int, device):
    user = torch.from_numpy(train_coo.row).long()
    item = torch.from_numpy(train_coo.col).long()
    item_offset = item + num_users
    indices = torch.cat(
        [
            torch.stack([user, item_offset], dim=0),
            torch.stack([item_offset, user], dim=0),
        ],
        dim=1,
    )
    values = torch.ones(indices.shape[1], dtype=torch.float32)
    n_nodes = num_users + num_items
    deg = torch.zeros(n_nodes, dtype=torch.float32)
    deg.index_add_(0, indices[0], values)
    deg_inv_sqrt = torch.pow(deg, -0.5)
    deg_inv_sqrt[torch.isinf(deg_inv_sqrt)] = 0.0
    norm_values = deg_inv_sqrt[indices[0]] * values * deg_inv_sqrt[indices[1]]
    norm_adj = torch.sparse_coo_tensor(indices, norm_values, (n_nodes, n_nodes))
    return norm_adj.coalesce().to(device)

In [5]:
# Config
DATA_DIR_P5 = "phase5_outputs"
DATA_DIR_P4 = "phase4_outputs"
OUTPUT_DIR = os.path.join("outputs", f"lightgcn_baseline_{datetime.now().strftime('%Y%m%d_%H%M%S')}")
K = 10
EXAMPLE_USERS = 5
RANDOM_STATE = 42
USE_NDCG = True

EMBEDDING_SIZE = 64
N_LAYERS = 3
LEARNING_RATE = 0.001
REG_WEIGHT = 1e-4
EPOCHS = 100
BATCH_SIZE = 2048
LOG_EVERY = 10

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Device:", DEVICE)

Device: cpu


In [6]:
# Load data (same train/test split as other notebooks)
train_interactions = load_sparse_matrix(os.path.join(DATA_DIR_P5, "train_interactions.npz"))
test_interactions = load_sparse_matrix(os.path.join(DATA_DIR_P5, "test_interactions.npz"))

train_coo = train_interactions.tocoo()
num_users, num_items = train_interactions.shape
norm_adj = build_norm_adj(train_coo, num_users, num_items, DEVICE)

user_pos_items = [train_interactions[user].indices for user in range(num_users)]
user_pos_sets = [set(items.tolist()) for items in user_pos_items]
users_with_train = np.where(train_interactions.getnnz(axis=1) > 0)[0]

print("Train shape:", train_interactions.shape)
print("Test shape:", test_interactions.shape)
print("Users with train interactions:", users_with_train.size)

book2id_path = os.path.join(DATA_DIR_P4, "book2id.json")
id2book = {}
if os.path.exists(book2id_path):
    id2book = {v: k for k, v in load_json(book2id_path).items()}

user2id_path = os.path.join(DATA_DIR_P4, "user2id.json")
id2user = {}
if os.path.exists(user2id_path):
    id2user = {v: k for k, v in load_json(user2id_path).items()}

Train shape: (79153, 20700)
Test shape: (79153, 20700)
Users with train interactions: 79153


In [8]:
# Train LightGCN (implicit BPR)
class LightGCN(torch.nn.Module):
    def __init__(self, num_users, num_items, embedding_dim, n_layers):
        super().__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.n_layers = n_layers
        self.user_embedding = torch.nn.Embedding(num_users, embedding_dim)
        self.item_embedding = torch.nn.Embedding(num_items, embedding_dim)
        torch.nn.init.normal_(self.user_embedding.weight, std=0.1)
        torch.nn.init.normal_(self.item_embedding.weight, std=0.1)

    def forward(self, norm_adj):
        all_embeddings = torch.cat([self.user_embedding.weight, self.item_embedding.weight], dim=0)
        embeddings = [all_embeddings]
        for _ in range(self.n_layers):
            all_embeddings = torch.sparse.mm(norm_adj, all_embeddings)
            embeddings.append(all_embeddings)
        final_embeddings = torch.mean(torch.stack(embeddings, dim=0), dim=0)
        user_final, item_final = torch.split(final_embeddings, [self.num_users, self.num_items])
        return user_final, item_final

def sample_batch(rng, users_with_train, user_pos_items, user_pos_sets, num_items, batch_size):
    batch_users = rng.choice(users_with_train, size=batch_size, replace=True)
    pos_items = np.empty(batch_size, dtype=np.int64)
    neg_items = np.empty(batch_size, dtype=np.int64)
    for idx, user_id in enumerate(batch_users):
        pos_items[idx] = rng.choice(user_pos_items[user_id])
        while True:
            candidate = rng.randint(0, num_items)
            if candidate not in user_pos_sets[user_id]:
                neg_items[idx] = candidate
                break
    return batch_users, pos_items, neg_items

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
rng = np.random.RandomState(RANDOM_STATE)

model = LightGCN(num_users, num_items, EMBEDDING_SIZE, N_LAYERS).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

for epoch in range(1, EPOCHS + 1):
    model.train()
    batch_users, pos_items, neg_items = sample_batch(
        rng,
        users_with_train,
        user_pos_items,
        user_pos_sets,
        num_items,
        BATCH_SIZE,
    )

model_path = os.path.join(OUTPUT_DIR, "lightgcn_model.pt")
torch.save(model.state_dict(), model_path)
print(f"Model saved to: {model_path}")

Model saved to: outputs/lightgcn_baseline_20260525_144022/lightgcn_model.pt


In [9]:
# Train LightGCN (implicit BPR)
class LightGCN(torch.nn.Module):
    def __init__(self, num_users, num_items, embedding_dim, n_layers):
        super().__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.n_layers = n_layers
        self.user_embedding = torch.nn.Embedding(num_users, embedding_dim)
        self.item_embedding = torch.nn.Embedding(num_items, embedding_dim)
        torch.nn.init.normal_(self.user_embedding.weight, std=0.1)
        torch.nn.init.normal_(self.item_embedding.weight, std=0.1)

    def forward(self, norm_adj):
        all_embeddings = torch.cat([self.user_embedding.weight, self.item_embedding.weight], dim=0)
        embeddings = [all_embeddings]
        for _ in range(self.n_layers):
            all_embeddings = torch.sparse.mm(norm_adj, all_embeddings)
            embeddings.append(all_embeddings)
        final_embeddings = torch.mean(torch.stack(embeddings, dim=0), dim=0)
        user_final, item_final = torch.split(final_embeddings, [self.num_users, self.num_items])
        return user_final, item_final

def sample_batch(rng, users_with_train, user_pos_items, user_pos_sets, num_items, batch_size):
    batch_users = rng.choice(users_with_train, size=batch_size, replace=True)
    pos_items = np.empty(batch_size, dtype=np.int64)
    neg_items = np.empty(batch_size, dtype=np.int64)
    for idx, user_id in enumerate(batch_users):
        pos_items[idx] = rng.choice(user_pos_items[user_id])
        while True:
            candidate = rng.randint(0, num_items)
            if candidate not in user_pos_sets[user_id]:
                neg_items[idx] = candidate
                break
    return batch_users, pos_items, neg_items

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
rng = np.random.RandomState(RANDOM_STATE)

model = LightGCN(num_users, num_items, EMBEDDING_SIZE, N_LAYERS).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

for epoch in range(1, EPOCHS + 1):
    model.train()
    batch_users, pos_items, neg_items = sample_batch(
        rng,
        users_with_train,
        user_pos_items,
        user_pos_sets,
        num_items,
        BATCH_SIZE,
    )
    batch_users_t = torch.from_numpy(batch_users).long().to(DEVICE)
    pos_items_t = torch.from_numpy(pos_items).long().to(DEVICE)
    neg_items_t = torch.from_numpy(neg_items).long().to(DEVICE)

    user_emb, item_emb = model(norm_adj)
    u = user_emb[batch_users_t]
    pos = item_emb[pos_items_t]
    neg = item_emb[neg_items_t]

    pos_scores = (u * pos).sum(dim=1)
    neg_scores = (u * neg).sum(dim=1)
    bpr_loss = -torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-8).mean()
    reg_loss = (u.norm(2).pow(2) + pos.norm(2).pow(2) + neg.norm(2).pow(2)) / BATCH_SIZE
    loss = bpr_loss + REG_WEIGHT * reg_loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % LOG_EVERY == 0 or epoch == 1:
        print(f"Epoch {epoch:03d}: loss={loss.item():.6f}")

model_path = os.path.join(OUTPUT_DIR, "lightgcn_model.pt")
torch.save(model.state_dict(), model_path)
print(f"Model saved to: {model_path}")

Epoch 001: loss=0.692197
Epoch 010: loss=0.691804
Epoch 020: loss=0.689212
Epoch 030: loss=0.681347
Epoch 040: loss=0.666388
Epoch 050: loss=0.641409
Epoch 060: loss=0.603660
Epoch 070: loss=0.572580
Epoch 080: loss=0.525960
Epoch 090: loss=0.504855
Epoch 100: loss=0.472157
Model saved to: outputs/lightgcn_baseline_20260525_144022/lightgcn_model.pt


In [11]:
# Evaluate + save outputs
model.eval()
with torch.no_grad():
    user_emb, item_emb = model(norm_adj)
    user_emb = user_emb.cpu().numpy()
    item_emb = item_emb.cpu().numpy()
    item_emb_t = item_emb.T

def score_fn(user_id: int) -> np.ndarray:
    return user_emb[user_id] @ item_emb_t

metrics = evaluate_ranking(
    score_fn,
    train_interactions,
    test_interactions,
    k=K,
    use_ndcg=USE_NDCG,
 )

print("LightGCN metrics:")
for key, value in metrics.items():
    print(f"  {key}: {value}")

with open(os.path.join(OUTPUT_DIR, "metrics.json"), "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

with open(os.path.join(OUTPUT_DIR, "metrics.csv"), "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(metrics.keys()))
    writer.writeheader()
    writer.writerow(metrics)

# Recommendation examples
users_with_test = np.where(test_interactions.getnnz(axis=1) > 0)[0]
if users_with_test.size > 0:
    sampled_users = rng.choice(
        users_with_test,
        size=min(EXAMPLE_USERS, users_with_test.size),
        replace=False,
    )

LightGCN metrics:
  k: 10.0
  users_evaluated: 79153.0
  precision_at_k: 0.05604209568809774
  recall_at_k: 0.16501741079717064
  ndcg_at_k: 0.13771046367820244


In [12]:
# Save a compact CSV summary for quick comparison
summary_path = os.path.join(OUTPUT_DIR, "metrics_summary.csv")
summary_row = {
    "run_id": os.path.basename(OUTPUT_DIR),
    "k": metrics.get("k"),
    "users_evaluated": metrics.get("users_evaluated"),
    "precision_at_k": metrics.get("precision_at_k"),
    "recall_at_k": metrics.get("recall_at_k"),
    "ndcg_at_k": metrics.get("ndcg_at_k", None),
}

write_header = not os.path.exists(summary_path)
with open(summary_path, "a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(summary_row.keys()))
    if write_header:
        writer.writeheader()
    writer.writerow(summary_row)

print(f"Summary saved to: {summary_path}")

Summary saved to: outputs/lightgcn_baseline_20260525_144022/metrics_summary.csv
